# evaluation

Evaluates one checkpoint on the held-out test split and writes a `results/*.json`.

Set the base model size via `task basemodel` and seed via `task dataset`. Toggle `EVAL_MODE` in the config cell:
- `EVAL_MODE="local"` (default) loads the **trained** checkpoint from `models/<model>_seed<seed>/`.
- `EVAL_MODE="hf"` loads the model ID set in `MODELID` (edit it directly in the config cell) — any HF repo, including your own fine-tuned uploads.

`GEN_MAX_NEW` is a **frozen** study constant — derived once from the real test set, hardcoded,
and identical across every size and seed.

Reports exact match (discontinuous) and token-sim (continuous), plus EOS-emission rate and
length ratio as diagnostics. Aggregate `results/*.json` in a separate plotting step.

In [ ]:
# === config ===
import json, os, math, difflib, re
from pathlib import Path
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from dotenv import load_dotenv

PROJECT_ROOT = Path("../..").resolve()
load_dotenv(PROJECT_ROOT / ".env")
MODEL_NAME = os.environ["BASEMODEL"].split("/")[-1]   # e.g. "Qwen2.5-Coder-0.5B"
SEED       = int(os.environ["SEED"])
REVISION   = os.environ["REVISION"]

# ---- toggle: "local" = local trained checkpoint, "hf" = any HF model id ----
EVAL_MODE = "local"   # "local" | "hf"
MODELID   = "notoh/qwen-coder-compilers"   # used only when EVAL_MODE="hf"

MODEL_DIR   = PROJECT_ROOT / "models" / f"{MODEL_NAME}_seed{SEED}"
RESULTS_DIR = PROJECT_ROOT / "results"; RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if EVAL_MODE == "hf":
    LOAD_FROM   = MODELID
    RESULT_STEM = f"{MODELID.split('/')[-1]}_hf"
else:  # "local"
    LOAD_FROM   = str(MODEL_DIR)
    RESULT_STEM = f"{MODEL_NAME}_seed{SEED}"
    assert MODEL_DIR.exists(), f"no checkpoint at {MODEL_DIR} -- train it first"

# ---- FROZEN eval constants: identical across EVERY size and seed ----
# Derive GEN_MAX_NEW ONCE from the real test set, then hardcode. Recomputing per run would
# make it differ by eval set and break cross-size comparability.
GEN_MAX_NEW = 5632          # p99 of the 68-example test set, rounded up to 256
EVAL_BATCH  = 8             # lower to 2-4 at 7B
MAX_EVAL    = None          # None = full test split

REPO   = "notoh/exebench_llvm_o3"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device      :", device)
print("eval mode   :", EVAL_MODE)
print("load from   :", LOAD_FROM)
print("result      :", RESULT_STEM + ".json")

In [ ]:
# === load model + the prompt it was trained with ===
cfg = {}
cfg_path = MODEL_DIR / "run_config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
PROMPT_TMPL = cfg.get("prompt_tmpl", "Compile the following C to LLVM IR at -O3:\n{c}\n---\n")
print("prompt template:", repr(PROMPT_TMPL))

tok = AutoTokenizer.from_pretrained(LOAD_FROM)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LOAD_FROM, dtype=torch.bfloat16 if device == "cuda" else torch.float32)
model.to(device).eval()
model.config.use_cache = True          # fast generation

test_ds = load_dataset(REPO, revision=REVISION)["test"]
if MAX_EVAL:
    test_ds = test_ds.select(range(min(MAX_EVAL, len(test_ds))))
print(f"eval examples: {len(test_ds)}")

In [ ]:
# === sanity: does the FROZEN GEN_MAX_NEW cover the targets? (warn only -- never auto-change) ===
tgt_lens = [len(tok(r["o3_ir"], add_special_tokens=False)["input_ids"]) for r in test_ds]
p50, p95, p99 = (int(np.percentile(tgt_lens, q)) for q in (50, 95, 99))
print(f"target tokens  p50={p50}  p95={p95}  p99={p99}  max={max(tgt_lens)}")
over = sum(l > GEN_MAX_NEW for l in tgt_lens)
print(f"GEN_MAX_NEW={GEN_MAX_NEW}  ({over}/{len(tgt_lens)} targets exceed it)")
if GEN_MAX_NEW < p99:
    print("WARNING: ceiling < p99 -- long targets truncate. Keep it frozen anyway for comparability.")

In [ ]:
# === metrics + batched eval ===
def normalize(s):
    return "\n".join(l.rstrip() for l in s.strip().splitlines())

def token_sim(a, b):           # continuous metric in [0, 1]
    ta = tok(a, add_special_tokens=False)["input_ids"]
    tb = tok(b, add_special_tokens=False)["input_ids"]
    return difflib.SequenceMatcher(None, ta, tb).ratio()

tok.padding_side = "left"      # correct decoder-only batched generation
exact, sims, len_ratio, eos_emitted = 0, [], [], 0
n = len(test_ds)
for start in range(0, n, EVAL_BATCH):
    recs    = [test_ds[k] for k in range(start, min(start + EVAL_BATCH, n))]
    prompts = [PROMPT_TMPL.format(c=r["c_source"]) for r in recs]
    enc = tok(prompts, add_special_tokens=False, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=GEN_MAX_NEW,
                             do_sample=False, pad_token_id=tok.pad_token_id)
    new = gen[:, enc["input_ids"].shape[1]:]
    for j, r in enumerate(recs):
        ids_out = new[j]
        if (ids_out == tok.eos_token_id).any():
            eos_emitted += 1
        pred = tok.decode(ids_out, skip_special_tokens=True)
        ref  = r["o3_ir"]
        if normalize(pred) == normalize(ref):
            exact += 1
        sims.append(token_sim(pred, ref))
        rl = len(tok(pred, add_special_tokens=False)["input_ids"])
        tl = len(tok(ref,  add_special_tokens=False)["input_ids"])
        len_ratio.append(rl / max(tl, 1))
    print(f"  {min(start + EVAL_BATCH, n)}/{n} done")

results = {
    "model": MODEL_NAME, "base": EVAL_BASE, "seed": (None if EVAL_BASE else SEED), "n": n,
    "exact_match":    exact / n,                 # discontinuous
    "token_sim_mean": float(np.mean(sims)),      # continuous
    "token_sim_std":  float(np.std(sims)),
    "eos_rate":       eos_emitted / n,           # % that stopped on their own
    "len_ratio_mean": float(np.mean(len_ratio)), # 1.0 = right length; >>1 = over-generating
    "gen_max_new":    GEN_MAX_NEW,
}
print("\n=== results ===")
for k, v in results.items():
    print(f"  {k:16s}: {v}")

out = RESULTS_DIR / f"{RESULT_STEM}.json"
out.write_text(json.dumps(results, indent=2))
print("\nsaved ->", out)